# Testes das classes base

Notebook pra validar `AgentRole`, `AgentContext`, `AgentResponse`, `BaseAgent` e `AgenteInvestimentos`, usando um LLM real da OpenAI (`ChatOpenAI`) via LangChain. Requer um `.env` com `OPENAI_API_KEY` preenchida.

## 0. Setup — deixar o projeto importável

In [ ]:
import sys
from pathlib import Path

# Sobe um nível (de notebooks/ para a raiz do projeto) e adiciona ao path,
# assim `from src...` funciona igual funcionaria rodando a partir da raiz.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Raiz do projeto: {project_root}")

## 1. `AgentRole` — testando o Enum

In [ ]:
from src.core.schemas import AgentRole

print(AgentRole.INVESTIMENTOS)
print(AgentRole.INVESTIMENTOS.value)

# Comparação direta com string funciona por causa do (str, Enum)
print(AgentRole.INVESTIMENTOS == "investimentos")

# Converter uma string vinda de fora (ex: JSON) para o Enum
print(AgentRole("dividas"))

# String inválida deve dar erro -- é a proteção contra typo
try:
    AgentRole("investimento")  # sem o 's' -- typo proposital
except ValueError as e:
    print(f"Erro esperado: {e}")

## 2. `AgentContext` — testando o schema de entrada

In [ ]:
from src.core.schemas import AgentContext

contexto = AgentContext(
    session_id="sessao-teste-001",
    user_message="O que é tesouro direto?",
)

print(contexto)
print()
# history e metadata vieram vazios por padrão -- confirma que default_factory funcionou
print(f"history: {contexto.history}")
print(f"metadata: {contexto.metadata}")

In [ ]:
# Validação de tipo do Pydantic em ação: isso deve dar erro,
# porque session_id precisa ser string
from pydantic import ValidationError

try:
    AgentContext(session_id=123, user_message="teste")
except ValidationError as e:
    print(f"Erro esperado de validação:\n{e}")

## 3. `BaseAgent` — confirmando que não dá pra instanciar direto

In [ ]:
from src.agents.base import BaseAgent

try:
    BaseAgent(llm=None)
except TypeError as e:
    print(f"Erro esperado (classe abstrata): {e}")

## 4. `AgenteInvestimentos` com a OpenAI de verdade

Carregamos a chave do `.env` e instanciamos um `ChatOpenAI` real. Repare que o `AgenteInvestimentos` não muda em nada -- é a mesma classe, só o objeto `llm` injetado no construtor é que é real agora em vez de falso (injeção de dependência).

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(project_root / ".env")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY não encontrada -- confira se o .env existe na raiz "
    "do projeto e está preenchido."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [ ]:
from src.agents.investimentos import AgenteInvestimentos
from src.core.schemas import AgentContext

agente = AgenteInvestimentos(llm=llm)

contexto = AgentContext(
    session_id="sessao-teste-001",
    user_message="O que é tesouro direto?",
)

resposta = agente.run(contexto)

print()
print(f"agent: {resposta.agent}")
print(f"content: {resposta.content}")
print(f"requires_review: {resposta.requires_review}")
print(f"created_at: {resposta.created_at}")

## 5. Testando com histórico de conversa

Confirma que `_build_messages` está juntando system prompt + histórico + mensagem atual corretamente.

In [ ]:
contexto_com_historico = AgentContext(
    session_id="sessao-teste-001",
    user_message="E fundos imobiliários, o que são?",
    history=[
        {"role": "user", "content": "O que é tesouro direto?"},
        {"role": "assistant", "content": "É uma plataforma para comprar títulos públicos."},
    ],
)

resposta2 = agente.run(contexto_com_historico)
print()
print(f"content: {resposta2.content}")

## Próximos passos

- Se quiser trocar de provedor no futuro (Anthropic ou Groq), troque só a célula que instancia `llm` por `ChatAnthropic(model="...")` ou `ChatGroq(model="...")` -- nada no `AgenteInvestimentos` muda, é exatamente esse o ganho da injeção de dependência.
- Fique de olho no consumo de créditos da OpenAI ao rodar este notebook várias vezes -- cada execução da seção 4 e 5 é uma chamada real de API.